# 03 · Fragmentation EDA — RQ1

**Research question:** How much GPU capacity is lost to resource fragmentation,
and how does fragmentation vary across workload types, priority classes, server
tiers, and time?

**Metric (Phase 3a — allocation-based lower bound):**
`frag_rate = 1 - (total_requested_gpu_hours / total_capacity_gpu_hours)` per server.
This measures capacity never assigned to any job. Utilization-based waste
(allocated but underutilised GPUs) requires pod_hourly data (Phase 3b).

**Prerequisites:** `make preprocess` completed for v2026.

In [5]:
import sys
print(sys.executable)
print(sys.version)


c:\Users\monis\Documents\Gpu\.venv\Scripts\python.exe
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (
    INTERIM_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR,
    DUCKDB_MEMORY_LIMIT, DUCKDB_THREADS,
)
from src.viz.style import set_ieee_style
import src.analysis.fragmentation as frag
import src.analysis.wasted_hours as wh
from src.viz.figures import (
    fig_frag_cdf_by_spec, fig_frag_heatmap_day_hour,
    fig_frag_by_priority, fig_frag_by_tier,
)

set_ieee_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
print('Ready')

## Headline numbers — total waste

Total wasted GPU-hours and energy for all three TDP scenarios.

In [ ]:
print('=== Waste Report (allocation-based lower bound) ===')
for r in wh.waste_report(con):
    print(r)

gpu_hours = wh.total_unallocated_gpu_hours(con)
kwh_mixed = wh.wasted_kwh(gpu_hours, tdp_watts=wh.MIXED_TDP_W)
co2_g     = wh.wasted_co2eq(kwh_mixed)

print(f'\nHeadline (mixed fleet, cn-beijing grid):')
print(f'  Unallocated GPU-hours : {gpu_hours:,.0f}')
print(f'  Wasted energy         : {kwh_mixed:,.1f} kWh')
print(f'  CO₂eq                 : {co2_g/1e9:,.2f} tonnes')

In [ ]:
print('=== Waste by workload type ===')
display(wh.breakdown_by_workload(con))

print('\n=== Waste by priority ===')
display(wh.breakdown_by_priority(con))

## Load analysis datasets

In [ ]:
print('Loading per-server fragmentation …')
frag_df = frag.compute_frag_rate(con)
print(f'  {len(frag_df):,} servers  |  frag_rate range: [{frag_df["frag_rate"].min():.3f}, {frag_df["frag_rate"].max():.3f}]')
print(f'  Server tiers: {frag_df["server_tier"].value_counts().to_dict()}')

print('\nLoading priority aggregation …')
priority_df = frag.aggregate_by_priority(con)
display(priority_df)

print('\nLoading workload aggregation …')
workload_df = frag.aggregate_by_workload_bucket(con)
display(workload_df)

## Figure 1 — CDF of server allocation-gap by GPU spec

Shows which GPU types have the highest fraction of capacity never allocated to jobs.

In [ ]:
p = fig_frag_cdf_by_spec(frag_df)
print(f'Saved: {p}')
plt.show()

## Figure 2 — Fleet GPU availability heatmap: day × hour-of-day

Shows fleet capacity dynamics — maintenance windows, provisioning patterns, weekly cycles.
Values are total GPUs available, not fragmentation rate (that needs pod_hourly).

In [ ]:
hmap_df = frag.day_hour_heatmap(con)
p = fig_frag_heatmap_day_hour(hmap_df)
print(f'Saved: {p}')
plt.show()

# Days with lowest fleet availability (maintenance windows / failures?)
long = hmap_df.reset_index().melt(id_vars='day', var_name='hour_int', value_name='gpus')
print('\nTop-5 lowest-availability (day, hour) cells:')
display(long.nsmallest(5, 'gpus')[['day','hour_int','gpus']])

## Figure 3 — Fragmentation by priority class

In [ ]:
p = fig_frag_by_priority(priority_df)
print(f'Saved: {p}')
plt.show()

print('\nGPU-hours by priority class:')
display(frag.aggregate_by_priority(con))

## Figure 4 — Fragmentation by server tier

In [ ]:
p = fig_frag_by_tier(frag_df)
print(f'Saved: {p}')
plt.show()

print('\nAllocation-gap stats by server tier:')
display(frag.aggregate_by_server_tier(con))

## Table 1 draft — workload bucket × frag rate percentiles

This becomes Table 1 in Section 4 of the paper.

In [ ]:
table1 = frag.aggregate_by_workload_bucket(con)
print('Table 1: GPU-hours by workload type')
display(table1)
table1.to_csv(TABLE_DIR / 'table1_frag_by_workload.csv', index=False)
print(f'Saved to {TABLE_DIR}/table1_frag_by_workload.csv')

## Diurnal and longitudinal profiles

Supports the narrative in Section 4.2.

In [ ]:
hod    = frag.aggregate_by_hour_of_day(con)
day_df = frag.aggregate_by_day(con)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))

axes[0].fill_between(hod['hour_int'], hod['p50_gpus'] / 1000,
                     alpha=0.3, color='#0072B2', label='p50')
axes[0].fill_between(hod['hour_int'], hod['p95_gpus'] / 1000,
                     alpha=0.15, color='#D55E00', label='p95')
axes[0].plot(hod['hour_int'], hod['mean_gpus'] / 1000,
             color='#0072B2', lw=1.0, label='mean')
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Fleet GPUs (K)')
axes[0].set_title('Diurnal fleet capacity profile')
axes[0].legend()

axes[1].fill_between(day_df['day'], day_df['min_gpus'] / 1000,
                     day_df['max_gpus'] / 1000,
                     alpha=0.2, color='#0072B2', label='min–max')
axes[1].plot(day_df['day'], day_df['mean_gpus'] / 1000,
             color='#0072B2', lw=0.8, label='mean')
axes[1].set_xlabel('Trace day')
axes[1].set_ylabel('Fleet GPUs (K)')
axes[1].set_title('Longitudinal fleet size (185 days)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_diurnal_longitudinal.pdf')
plt.show()

## v2023 validation — cross-version comparison

Run the same analysis on the v2023 trace (after `make preprocess` for v2023)
and compare fragmentation distributions to check whether v2026 is meaningfully
different or consistent with the earlier trace.

**Expected outcomes:**
- **Similar shape** → fragmentation is a structural property of the scheduler
  (consistent finding across versions).
- **v2026 higher** → cluster has grown faster than scheduling improvements;
  interesting trend to report in Section 7.
- **v2026 lower** → scheduler improvements in v2026; noteworthy if true.

In [ ]:
from pathlib import Path

# v2023 uses different processed dir (pipeline writes to data/processed/ when --trace v2023)
# For now just skip gracefully — will be enabled in Phase 3b after v2023 download
eff_v2023 = PROCESSED_DIR.parent / 'processed_v2023' / 'server_allocation_efficiency.parquet'

if not eff_v2023.exists():
    print('v2023 processed data not found.')
    print('Run: python -m src.preprocess.pipeline --trace v2023')
else:
    frag_v2023 = frag.compute_frag_rate(con, eff_src=eff_v2023)

    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    for label, df, color in [
        ('v2026 (155K GPUs)', frag_df,    '#0072B2'),
        ('v2023 (6.2K GPUs)', frag_v2023, '#E69F00'),
    ]:
        vals = df['frag_rate'].dropna().sort_values()
        cdf  = np.arange(1, len(vals)+1) / len(vals)
        ax.plot(vals, cdf, label=label, color=color, lw=1.0)

    ax.set_xlabel('Server allocation gap')
    ax.set_ylabel('CDF')
    ax.set_title('v2023 vs v2026 allocation gap (sanity check)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_v2023_v2026_comparison.pdf')
    plt.show()

    from scipy.stats import ks_2samp
    stat, pval = ks_2samp(
        frag_df['frag_rate'].dropna(),
        frag_v2023['frag_rate'].dropna(),
    )
    print(f'KS statistic: {stat:.4f}  p-value: {pval:.4e}')
    if pval < 0.05:
        print('  → Distributions differ (worth reporting in §7).')
    else:
        print('  → Cannot reject that distributions are the same.')